# Broker and Informed Trader

In this notebook, we compare the learned and exact equilibria of the broker-trader game of Wu and Jaimungal (2024),
without transient impact.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import BrokerTrader

## 1. The game

We denote by $z$ the price, by $\alpha$ a mean-reverting signal, and by $Q^I$ and $Q^B$ the inventories of the
informed trader and of the broker. The trader trades with the broker at rate $\eta$, and the broker trades on the
exchange at rate $\nu$. These processes follow

$$
\begin{aligned}
dz_t &= \alpha_t\,dt + \sigma\,dW_t, \\[4pt]
d\alpha_t &= -\kappa\,\alpha_t\,dt + \sigma_\alpha\,dB_t, \\[4pt]
dQ^I_t &= \eta_t\,dt, \qquad dQ^B_t = (\nu_t - \eta_t)\,dt ,
\end{aligned}
$$

where $W$ and $B$ are independent Brownian motions. The trader observes $(\alpha, Q^I, z)$. The broker observes the
price $z$ only, so the broker's information is contained in the trader's. The trader and the broker maximize

$$
\begin{aligned}
J^I &= \mathbb{E}\Big[ \int_0^T \big( Q^I_t\,\alpha_t - b\,\eta_t^2 - r_I\,(Q^I_t)^2 \big)\,dt - \psi\,(Q^I_T)^2 \Big], \\[4pt]
J^B &= \mathbb{E}\Big[ \int_0^T \big( Q^B_t\,\alpha_t - a\,\nu_t^2 - r_B\,(Q^B_t)^2 \big)\,dt - \phi\,(Q^B_T)^2 \Big].
\end{aligned}
$$

## 2. Exact equilibrium

The reward of the trader does not depend on the broker's control. Thus the trader's control solves a discrete LQR
problem in $(\alpha, Q^I)$. The broker estimates $(\alpha, Q^I)$ from the price by a discrete Kalman filter. The
broker's control then solves a discrete LQR problem in the estimates of $(Q^B, \alpha, Q^I)$.

## 3. Learned and exact equilibria

The learned method is described in the README, here with 5,000 paths and damping $\rho = 0.7$. The figure compares
the learned and exact controls on three validation paths.

In [ ]:
game = BrokerTrader()
solver = Solver(game, damping=0.7, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(10, reference=exact)
print(f'relative L2 error: trader {relative_l2(learned[0], exact[0]):.3f}, broker {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['informed trader', 'broker'])